
# Day 12 — GraphSAGE

> 目标：把 GraphSAGE 真正放进昨天学过的 **Message + Aggregation** 框架里，并理解它为什么比 shallow embedding / 纯 lookup 更有 **inductive potential**。

今天主线：

1. 从 GCN 过渡到 GraphSAGE
2. 理解 two-stage aggregation
3. 理解 `self` 和 `neighbors` 为什么分开
4. 手搓最简 Mean GraphSAGE
5. 和 GCN 做结构对比
6. 用 PyG `SAGEConv`
7. 做一个小型 node classification 实验
8. 建立 sampling + inductive 的直觉

---

## 今天必须真正理解

- GraphSAGE 的两阶段结构
- `AGG(neighbors)` 和 `CONCAT(self, neighbor_agg)` 分别做什么
- 为什么 GraphSAGE 不需要“每个 node 一套 embedding 参数”
- 为什么 shared parameters 带来 inductive potential
- GraphSAGE 和 GCN 的主要结构差异
- PyG `SAGEConv(x, edge_index)` 在概念上对应什么


## Formula rendering check

$$
h_v^{(l)}
=
\sigma\left(
\sum_{u\in N(v)}
\alpha_{vu}W^{(l)}h_u^{(l-1)}
\right)
$$

$$
h_{N(v)}^{(l)}
=
\frac{1}{|N(v)|}
\sum_{u\in N(v)}
h_u^{(l-1)}
$$



# 1. 从昨天的 GCN 开始

昨天我们学过的 GCN 可以粗略写成：

$$
h_v^{(l)}
=
\sigma\left(
\sum_{u \in N(v)\cup\{v\}}
\alpha_{vu} W^{(l)}h_u^{(l-1)}
\right)
$$

它把：

```text
self + neighbors
```

通过统一的图归一化权重一起做 message passing。

GraphSAGE 的思路不一样：

> **先聚合 neighbors，再把 neighbor representation 和 self representation 合起来。**



# 2. GraphSAGE 的 two-stage aggregation

课件里的核心形式：

$$
h_{N(v)}^{(l)}
=
\mathrm{AGG}\left(
\{h_u^{(l-1)},\, u\in N(v)\}
\right)
$$

先得到一个邻居整体表示：

```text
neighbors
    ↓
AGG
    ↓
h_N(v)
```

然后：

$$
h_v^{(l)}
=
\sigma
\left(
W^{(l)}
\cdot
\mathrm{CONCAT}
\left(
h_v^{(l-1)},
h_{N(v)}^{(l)}
\right)
\right)
$$

也就是：

```text
self h_v --------┐
                  CONCAT → Linear → activation → new h_v
neighbors → AGG --┘
```

---

## 必须回答 1

为什么 GraphSAGE 不直接把 `self` 混进邻居里一起 mean，而是先：

```text
AGG(neighbors)
```

再：

```text
CONCAT(self, neighbor_agg)
```

？

你的回答：保留 self 与 neighborhood 的独立信息，使后续线性变换能够分别学习两者的重要性



# 3. GraphSAGE 中的 Message + Aggregation

这一点容易绕。

在 GraphSAGE 里，可以这样建立直觉：

### 邻居侧

每个 neighbor 的 representation：

$$
h_u^{(l-1)}
$$

先进入某种 aggregator。

例如最简单的 Mean：

$$
h_{N(v)}^{(l)}
=
\frac{1}{|N(v)|}
\sum_{u\in N(v)}
h_u^{(l-1)}
$$

### self + neighborhood

然后把：

$$
h_v^{(l-1)}
$$

和：

$$
h_{N(v)}^{(l)}
$$

拼起来：

$$
\mathrm{CONCAT}
\left(
h_v^{(l-1)},
h_{N(v)}^{(l)}
\right)
$$

最后通过共享的线性层：

$$
W^{(l)}
$$

得到新的 node representation。

---

一句话：

> **GraphSAGE 先把邻居压缩成一个固定长度向量，再和自己的 representation 合并。**



# 4. Mean / Pool / LSTM Aggregator

课件给了三种典型形式。

## Mean

$$
\mathrm{AGG}
=
\frac{1}{|N(v)|}
\sum_{u\in N(v)}
h_u
$$

最容易理解，也最适合我们今天手搓。

## Pool

先对每个 neighbor 做一个可学习变换：

$$
\mathrm{MLP}(h_u)
$$

再做 symmetric function，例如：

$$
\mathrm{Mean}
\quad\text{or}\quad
\mathrm{Max}
$$

## LSTM

把 neighbors 按某种顺序送进 LSTM。

今天只需要知道它是原论文中的一种 aggregator 选择，**不用手搓**。

---

## 必须回答 2

为什么 Mean / Max 天然不依赖 neighbor ordering？

你的回答：


In [3]:

import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import networkx as nx

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

G = nx.Graph()
G.add_edges_from([
    (0, 1), (0, 2),
    (1, 3), (1, 4),
    (2, 5)
])

X = torch.tensor([
    [1.0, 0.0, 0.5],
    [0.5, 1.0, 0.0],
    [0.0, 1.0, 1.0],
    [1.0, 1.0, 0.0],
    [0.0, 0.5, 1.0],
    [1.0, 0.0, 1.0],
])

print("X shape:", X.shape)
print("neighbors of node 0:", list(G.neighbors(0)))


X shape: torch.Size([6, 3])
neighbors of node 0: [1, 2]



# 5. 手搓 Mean Aggregator

先只做这一件事：

> 给 node `v`，把它所有 neighbors 的 representation 取 mean。

如果：

```text
H[nbrs].shape = [m, d]
```

那么：

```python
H[nbrs].mean(dim=0)
```

输出：

```text
[d]
```

这里不是把整个 graph 压成一维。

而是：

> **每个 node 单独得到一个长度为 d 的 neighbor summary。**

所有 node 都做一遍以后，再 stack 回：

```text
[num_nodes, d]
```


In [4]:
def mean_aggregate_one_node(G, H, v):
    nbrs = list(G.neighbors(v))

    if len(nbrs) == 0:
        return torch.zeros(H.shape[1], dtype=H.dtype)

    return H[nbrs].mean(dim=0)

for v in G.nodes():
    print(
        f"node {v}:",
        mean_aggregate_one_node(G, X, v)
    )


node 0: tensor([0.2500, 1.0000, 0.5000])
node 1: tensor([0.6667, 0.5000, 0.5000])
node 2: tensor([1.0000, 0.0000, 0.7500])
node 3: tensor([0.5000, 1.0000, 0.0000])
node 4: tensor([0.5000, 1.0000, 0.0000])
node 5: tensor([0., 1., 1.])



## 必须回答 3

假设：

```text
H.shape = [6, 3]
degree(v) = 2
```

那么：

```text
H[v]
H[nbrs]
H[nbrs].mean(dim=0)
```

shape 分别是什么？

你的回答：[3], [2, 3], [3]



# 6. 手搓最简 Mean GraphSAGE Layer

我们实现：

$$
h_v'
=
\sigma
\left(
W\,
\mathrm{CONCAT}
\left(
h_v,
\mathrm{Mean}\{h_u:u\in N(v)\}
\right)
\right)
$$

注意：

如果输入维度是：

$$
d
$$

那么：

```text
self representation:     [d]
neighbor mean:           [d]
concat 后:               [2d]
```

所以 Linear 应该是：

```text
2 * in_dim -> out_dim
```


In [8]:
class ToyMeanGraphSAGE(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.linear = nn.Linear(2*in_dim, out_dim)

    def forward(self, G, H):
        out = []

        for v in G.nodes():
            h_self = H[v]
            h_nbrs = mean_aggregate_one_node(G, H, v)

            h_cat = torch.cat([h_self, h_nbrs], dim=0)
            h_new = self.linear(h_cat)
            h_new = F.relu(h_new)

            out.append(h_new)

        return torch.stack(out)


sage1 = ToyMeanGraphSAGE(in_dim=3, out_dim=4)
H1 = sage1(G, X)

print("X shape :", X.shape)
print("H1 shape:", H1.shape)
print(H1)


X shape : torch.Size([6, 3])
H1 shape: torch.Size([6, 4])
tensor([[0.0000, 0.1065, 0.4151, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.2237, 0.0000],
        [0.0000, 0.0031, 0.1905, 0.0000],
        [0.0000, 0.0726, 0.9229, 0.0000],
        [0.0000, 0.0693, 0.3962, 0.0000]], grad_fn=<StackBackward0>)



## 必须回答 4

为什么：

```python
self.linear = nn.Linear(2 * in_dim, out_dim)
```

而不是：

```python
nn.Linear(in_dim, out_dim)
```

？

你的回答：因为 GraphSAGE 会先把当前节点的表示 $h_v$ 和邻居聚合后的表示 $h_{N(v)}$ 做 CONCAT。两者的维度都是 in_dim，所以拼接后得到的向量维度是 2 * in_dim。因此后面的线性层输入维度必须写成 2 * in_dim



# 7. GraphSAGE vs GCN

现在把结构压成最重要的一张表。

| | GCN | GraphSAGE |
|---|---|---|
| 邻居处理 | normalized weighted sum | configurable aggregator |
| self | 常通过 self-loop 一起聚合 | 显式保留并和邻居表示合并 |
| aggregation | 由 GCN normalization 规定 | Mean / Pool / LSTM 等 |
| 参数 | 全图共享 | 全图共享 |
| 新节点潜力 | 有 | GraphSAGE 强调 inductive setting |
| 大图训练 | 可做 sparse message passing | 原始设计特别强调 neighbor sampling |

---

## 必须回答 5

GraphSAGE 比 shallow `nn.Embedding(num_nodes, d)` 更有 inductive potential 的根本原因是什么？

你的回答：Shallow embedding 学的是“每个节点自己的 embedding 参数”，而 GraphSAGE 学的是一个从 node features 和 neighborhood 到 embedding 的共享函数，因此可以应用于训练时未见过的新节点。



# 8. 什么叫 inductive？

假设训练时只有：

```text
node 0, 1, 2, 3, 4, 5
```

测试时来了一个新 node：

```text
node 6
```

如果我们知道：

- `x_6`
- `neighbors(6)`
- 邻居的 features / representations

那么 GraphSAGE 可以直接把新 node 放进同一套：

```text
AGGREGATE
→ CONCAT
→ shared Linear
```

规则里。

它不需要提前为：

```text
node 6
```

在 `nn.Embedding(num_nodes, d)` 里预留一行。

---

## 必须回答 6

为什么：

```text
shared parameters
```

是 GraphSAGE 可以泛化到新 node 的核心条件之一？

你的回答：Shared parameters 使 GraphSAGE 学到的是可复用的节点表示生成规则，因此能够应用到训练时未见过的新节点。



# 9. Neighbor Sampling 的直觉

这一部分来自我们原定 Day12 规划，用来补 GraphSAGE 最重要的工程直觉。

假设一个 node 有：

```text
100000 个 neighbors
```

如果每层都把所有 neighbors 全部展开：

```text
1-hop
→ 2-hop
→ 3-hop
```

computation graph 会快速爆炸。

GraphSAGE 的典型做法是：

```text
每层只 sample 固定数量 neighbors
```

比如：

```text
layer 1: sample 10
layer 2: each sample 5
```

这样计算规模更可控。

今天不要求你实现真实 mini-batch neighbor sampling。

只要记住：

> **sample + aggregate 是 GraphSAGE 的关键直觉之一。**



## 必须回答 7

为什么 neighbor sampling 主要是在解决：

```text
scalability
```

而不是“让模型变得 inductive”本身？

你的回答：Neighbor sampling 主要解决大图 GNN 中 neighborhood expansion 带来的计算和内存开销。它通过每层只采样固定数量邻居来近似完整的 neighborhood aggregation。GraphSAGE 的 inductive ability 主要来自共享的 encoder 和基于 node feature + neighborhood 的表示生成规则，而不是 sampling 本身。



# 10. GraphSAGE 的 optional L2 normalization

课件还提到，每层输出之后可以做：

$$
h_v
\leftarrow
\frac{h_v}{\|h_v\|_2}
$$

这样每个 embedding 的 L2 norm 都变成 1。

注意：

> **这是 optional，不是 GraphSAGE 必须步骤。**

今天知道作用即可：

- 控制 embedding scale
- 让不同 node representation 的尺度更一致


In [4]:
h = torch.tensor([3.0, 4.0])

print("before:", h)
print("norm before:", torch.norm(h))

h_normalized = F.normalize(h, p=2, dim=0)

print("after:", h_normalized)
print("norm after:", torch.norm(h_normalized))


before: tensor([3., 4.])
norm before: tensor(5.)
after: tensor([0.6000, 0.8000])
norm after: tensor(1.)



# 11. PyG：`SAGEConv`

真实项目里不会像前面一样 Python：

```python
for v in G.nodes():
```

逐 node 算。

PyG 提供：

```python
SAGEConv
```

直接使用：

```text
x
edge_index
```

完成 message passing。


In [5]:

from torch_geometric.datasets import KarateClub
from torch_geometric.nn import SAGEConv

dataset = KarateClub()
data = dataset[0]

print(data)
print("x:", data.x.shape)
print("edge_index:", data.edge_index.shape)
print("y:", data.y.shape)


D:\anaconda\envs\graph-learning\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


Data(x=[34, 34], edge_index=[2, 156], y=[34], train_mask=[34])
x: torch.Size([34, 34])
edge_index: torch.Size([2, 156])
y: torch.Size([34])


In [6]:
class PyGGraphSAGE(nn.Module):
    def __init__(self, in_dim, hidden_dim, num_classes):
        super().__init__()
        self.conv1 = SAGEConv(
            in_dim,
            hidden_dim,
            aggr='mean'
        )

        self.conv2 = SAGEConv(
            hidden_dim,
            num_classes,
            aggr='mean'
        )

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        return x

model = PyGGraphSAGE(
    dataset.num_features,
    16,
    dataset.num_classes
)

print(model)

PyGGraphSAGE(
  (conv1): SAGEConv(34, 16, aggr=mean)
  (conv2): SAGEConv(16, 4, aggr=mean)
)



## 必须回答 8

在：

```python
self.conv1(x, edge_index)
```

里面：

- `x` 提供什么？
- `edge_index` 提供什么？
- shared trainable parameters 在哪里？

你的回答：x 提供所有节点的 feature / current representation；edge_index 提供图的 connectivity，用于确定 message passing 的邻居关系；shared trainable parameters 存在于 self.conv1 这个 SAGEConv layer 内部，例如其中的线性变换权重 \(W\)。这些参数在所有节点之间共享，而不是每个 node 各有一套参数。



# 12. 用 KarateClub 跑一次 GraphSAGE node classification

和昨天 GCN 一样：

```text
整张图参与 forward
```

但：

```text
只有 train_mask 中的 nodes 参与 supervised loss
```


In [7]:
model = PyGGraphSAGE(
    dataset.num_features,
    16,
    dataset.num_classes
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.01,
    weight_decay=5e-4
)

for epoch in range(1,201):
    model.train()
    optimizer.zero_grad()

    logits = model(data.x, data.edge_index)

    loss = F.cross_entropy(
        logits[data.train_mask],
        data.y[data.train_mask]
    )

    loss.backward()
    optimizer.step()

    if epoch % 40 == 0:
        model.eval()

        with torch.no_grad():
            logits = model(
                data.x,
                data.edge_index
            )

            pred = logits.argmax(dim=1)
            acc = (pred == data.y).float().mean().item()

        print(
                f"epoch={epoch:03d}",
                f"loss={loss.item():.4f}",
                f"all-node acc={acc:.4f}"
            )

epoch=040 loss=0.0180 all-node acc=0.7353
epoch=080 loss=0.0028 all-node acc=0.6765
epoch=120 loss=0.0029 all-node acc=0.6176
epoch=160 loss=0.0028 all-node acc=0.6176
epoch=200 loss=0.0026 all-node acc=0.5882



# 13. GCN vs GraphSAGE：代码层面对照

GCN：

```python
self.conv1 = GCNConv(in_dim, hidden_dim)
```

GraphSAGE：

```python
self.conv1 = SAGEConv(
    in_dim,
    hidden_dim,
    aggr="mean"
)
```

两者 forward 都是：

```python
conv(x, edge_index)
```

但内部 rule 不同。

GCN：

```text
degree-normalized message passing
```

GraphSAGE：

```text
aggregate neighbors
+
combine self and neighborhood
```

---

## 必须回答 9

为什么两个模型在 PyG 里调用方式看起来几乎一样，但它们并不是同一个算法？

你的回答：虽然在 PyG 中 GCNConv(x, edge_index) 和 SAGEConv(x, edge_index) 的调用接口很相似，都是接收 node features 和 graph connectivity，但它们内部采用不同的 message passing 和 aggregation rule。GCNConv 使用 degree-normalized aggregation，并通常通过 self-loop 将自身和邻居一起聚合；SAGEConv 则先聚合邻居，再显式与 self representation 组合，因此它们并不是同一个算法。



# 14. 小实验：同一数据集对比 GCN vs GraphSAGE

今天不追求论文级 benchmark。

只做一件事：

> 在同一个 KarateClub split 上，比较 GCN 和 GraphSAGE 是否都能完成 semi-supervised node classification。


In [8]:

from torch_geometric.nn import GCNConv


class PyGGCN(nn.Module):
    def __init__(self, in_dim, hidden_dim, num_classes):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, num_classes)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        return x


def train_and_eval(model, epochs=200):
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=0.01,
        weight_decay=5e-4
    )

    for _ in range(epochs):
        model.train()
        optimizer.zero_grad()

        logits = model(
            data.x,
            data.edge_index
        )

        loss = F.cross_entropy(
            logits[data.train_mask],
            data.y[data.train_mask]
        )

        loss.backward()
        optimizer.step()

    model.eval()

    with torch.no_grad():
        logits = model(
            data.x,
            data.edge_index
        )

        pred = logits.argmax(dim=1)
        acc = (pred == data.y).float().mean().item()

    return loss.item(), acc


torch.manual_seed(SEED)
gcn = PyGGCN(
    dataset.num_features,
    16,
    dataset.num_classes
)

torch.manual_seed(SEED)
sage = PyGGraphSAGE(
    dataset.num_features,
    16,
    dataset.num_classes
)

gcn_loss, gcn_acc = train_and_eval(gcn)
sage_loss, sage_acc = train_and_eval(sage)

print("GCN       :", gcn_loss, gcn_acc)
print("GraphSAGE :", sage_loss, sage_acc)


GCN       : 0.010833668522536755 0.8235294222831726
GraphSAGE : 0.0016299554845318198 0.5588235259056091



## 必须回答 10

如果这次实验里 GCN accuracy 比 GraphSAGE 高，能不能说明：

> GCN 比 GraphSAGE 更好？

为什么？

你的回答：不能。这个结果只能说明在当前数据集、数据划分、超参数、随机初始化和训练设置下，本次运行中 GCN 的 accuracy 高于 GraphSAGE。单次实验不能证明 GCN 本身优于 GraphSAGE；要进行可靠比较，需要控制实验条件，并进行多次不同随机种子的实验，比较平均性能及波动。



# Day 12 最终知识图

```text
Day 9
shallow embedding
node id → lookup
        ↓
transductive limitation

Day 10
general GNN
Message + Aggregation
shared parameters

Day 11
GCN
self-loop
symmetric normalization
GCNConv

Day 12
GraphSAGE
        ↓
neighbors → AGG
        ↓
neighbor representation
        ↓
CONCAT(self, neighbors)
        ↓
shared Linear
        ↓
new node representation
        ↓
inductive potential

large graph
        ↓
neighbor sampling
        ↓
control computation graph size
```



# Day 12 最终自测

1. GraphSAGE 的 two-stage aggregation 是哪两步？
2. 为什么 `self` 和 neighbor aggregate 要分开保留？
3. Mean aggregator 为什么 permutation invariant？
4. 为什么 concat 后 Linear 的输入维度通常是 `2 * in_dim`？
5. GraphSAGE 和 GCN 的核心结构区别是什么？
6. 为什么 GraphSAGE 有 inductive potential？
7. neighbor sampling 主要解决什么问题？
8. sampling 和 inductive 是不是一回事？
9. optional L2 normalization 做了什么？
10. PyG `SAGEConv(x, edge_index)` 里 `x` 和 `edge_index` 各自承担什么角色？
11. 为什么 `SAGEConv` 和 `GCNConv` 调用形式相似，但算法不同？
12. 为什么一次小数据集实验不能证明哪个模型普遍更优？


下一步：

> **Day13：GAT**
